# Run Evaluation

Evaluate the custom deep research pipeline on quality, performance, and cost metrics.

## 1. Load Environment

In [ ]:
import os
import json
import time
import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)

ORCHESTRATOR_URL = f"http://localhost:{os.getenv('ORCHESTRATOR_PORT', '8100')}"
print(f"Orchestrator: {ORCHESTRATOR_URL}")
print("✅ Environment loaded")

## 2. Define Test Queries

In [ ]:
test_queries = [
    "What is the main architecture described in the paper?",
    "What are the key contributions and innovations?",
    "How does the system handle document parsing and conversion?",
    "What evaluation metrics are used and what are the results?",
    "What are the limitations and future work directions?",
]

print(f"Prepared {len(test_queries)} test queries")
for i, q in enumerate(test_queries, 1):
    print(f"  {i}. {q}")

## 3. Run Evaluation Pipeline

In [ ]:
def run_research_query(query: str) -> dict:
    """Execute a research query and collect metrics."""
    payload = {
        "jsonrpc": "2.0",
        "method": "message/send",
        "id": f"eval-{hash(query) % 10000}",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": query}],
            }
        },
    }

    start = time.time()
    try:
        resp = httpx.post(ORCHESTRATOR_URL, json=payload, timeout=180)
        latency = time.time() - start
        result = resp.json()

        # Extract response text
        text = ""
        if "result" in result:
            for artifact in result["result"].get("artifacts", []):
                for part in artifact.get("parts", []):
                    if part.get("kind") == "text":
                        text += part["text"]
            if not text and "message" in result["result"]:
                for part in result["result"]["message"].get("parts", []):
                    if part.get("kind") == "text":
                        text += part["text"]

        return {
            "query": query,
            "latency_s": latency,
            "response_length": len(text),
            "response": text,
            "success": bool(text),
            "error": None,
        }
    except Exception as e:
        return {
            "query": query,
            "latency_s": time.time() - start,
            "response_length": 0,
            "response": "",
            "success": False,
            "error": str(e),
        }


# Run all queries
results = []
for i, query in enumerate(test_queries, 1):
    print(f"Running query {i}/{len(test_queries)}: {query[:50]}...")
    result = run_research_query(query)
    results.append(result)
    status = "✅" if result["success"] else "❌"
    print(f"  {status} {result['latency_s']:.1f}s | {result['response_length']} chars")

print(f"\nCompleted {len(results)} queries")

## 4. Analyze Results

In [ ]:
successful = [r for r in results if r["success"]]
failed = [r for r in results if not r["success"]]

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)

print(f"\nSuccess rate: {len(successful)}/{len(results)} ({100*len(successful)/len(results):.0f}%)")

if successful:
    latencies = [r["latency_s"] for r in successful]
    lengths = [r["response_length"] for r in successful]

    print(f"\nLatency (successful queries):")
    print(f"  Mean:   {sum(latencies)/len(latencies):.1f}s")
    print(f"  Min:    {min(latencies):.1f}s")
    print(f"  Max:    {max(latencies):.1f}s")
    print(f"  Median: {sorted(latencies)[len(latencies)//2]:.1f}s")

    print(f"\nResponse length:")
    print(f"  Mean:   {sum(lengths)/len(lengths):.0f} chars")
    print(f"  Min:    {min(lengths)} chars")
    print(f"  Max:    {max(lengths)} chars")

if failed:
    print(f"\nFailed queries:")
    for r in failed:
        print(f"  - {r['query'][:50]}... : {r['error']}")

print("\n✅ Evaluation analysis complete")

## 5. Quality Scoring

Use the Reviewer agent to score each response.

In [ ]:
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "agents", "reviewer"))

from agents.reviewer.tools import score_quality

scores = []
for r in successful:
    score = score_quality(r["response"])
    scores.append(score)
    print(f"  Query: {r['query'][:40]}... → Score: {score}/10")

if scores:
    print(f"\nQuality Summary:")
    print(f"  Mean score:  {sum(scores)/len(scores):.1f}/10")
    print(f"  Min score:   {min(scores)}/10")
    print(f"  Max score:   {max(scores)}/10")
    passing = sum(1 for s in scores if s >= 7.0)
    print(f"  Passing (≥7): {passing}/{len(scores)}")

print("\n✅ Quality evaluation complete")

## 6. Summary

In [ ]:
print("Custom Deep Research — Evaluation Summary")
print("=" * 50)
print(f"\nArchitecture:")
print(f"  Platform:    Kagenti on OpenShift AI")
print(f"  Protocol:    A2A (Agent-to-Agent)")
print(f"  Framework:   LangGraph")
print(f"  Agents:      5 (orchestrator + 4 workers)")
print(f"  Vector DB:   PostgreSQL + pgvector")
print(f"  Doc Parser:  Docling")

if successful and scores:
    print(f"\nPerformance:")
    print(f"  Avg latency:  {sum(latencies)/len(latencies):.1f}s")
    print(f"  Success rate: {100*len(successful)/len(results):.0f}%")
    print(f"  Avg quality:  {sum(scores)/len(scores):.1f}/10")

print(f"\n✅ Lab complete — Custom Deep Research on RHOAI")